# Préparation DPO — UltraMedical-Preference (anglais)

Paires préférentielles (`prompt` / `chosen` / `rejected`) pour l'alignement DPO, à partir de
[TsinghuaC3I/UltraMedical-Preference](https://huggingface.co/datasets/TsinghuaC3I/UltraMedical-Preference) (licence MIT) :
réponses de plusieurs LLM, notées et classées par GPT-4.

- **Qualité plutôt que quantité** : une paire n'est gardée que si la préférence est **vérifiée** : par le corrigé
  pour les QCM (`chosen` juste, `rejected` fausse), par un écart de note net sinon. La taille est un plafond, pas un
  objectif.
- **Test validé cliniquement** : le `test` d'UltraMedical (Medical RewardBench), dont les préférences ont été **revues
  par des experts**, sert de jeu de test.
- **Orienté triage** : mêmes sources et mêmes critères que le SFT anglais (`projet_14.selection`) : patient décrit
  obligatoire, priorité aux cas avec constantes et vocabulaire d'urgence.
- **Séparé du SFT** : seules les questions attribuées au DPO par leur empreinte (`case_assignment`) sont utilisées ;
  le SFT anglais prend l'autre moitié. Un contrôle final le vérifie.
- **Équilibré sur la longueur** : autant de paires où la `chosen` est la plus longue que la plus courte (leçon du DPO
  v3, qui avait appris « la réponse la plus courte est la meilleure »).
- **Version 5** (essai B) : moins de paires (800 au plus, contre 1 400) et des réponses plus courtes (2 000
  caractères au plus). En v4, les longues réponses anglaises pilotaient l'apprentissage (environ 30 fois plus de
  tokens qu'une paire française), et leur style débordait sur les réponses françaises. Le test, revu par des
  experts, est inchangé.
- **Sortie** : `data/v5/dpo_ultramedical_{train,validation,test}.jsonl`, puis la version anonymisée dans `data/v5/anonymized/`.
  Le SFT reste en v4 (`data/v4`).

# Import

In [1]:
import re
from pathlib import Path

import pandas as pd
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer

from projet_14.conversation import SYSTEM_PROMPT_EN
from projet_14.mediqal import case_assignment, case_id
from projet_14.rgpd import run
from projet_14.selection import PATIENT_EN, triage_flags

OUT = Path("../data/v5")
SFT_DIR = Path("../data/v4/anonymized")  # données SFT (v4, inchangées)
STEM = "dpo_ultramedical"
SEED = 42
MODEL_NAME = "Qwen/Qwen3-1.7B-Base"  # pour compter les tokens avec le tokenizer du modèle entraîné

# Mêmes sources que le SFT anglais : vignettes cliniques (QCM d'examen) et patients
SOURCES = ["MedQA", "MedQA-Evol", "MedMCQA", "ChatDoctor"]
MCQ_SOURCES = ["MedQA", "MedQA-Evol", "MedMCQA"]  # bonne réponse (lettre) vérifiable dans les deux réponses
# Sources sans corrigé : seuils sur les notes GPT-4 (de 1 à 5)
MIN_CHOSEN_SCORE = 4.5
MIN_SCORE_GAP = 2

# Longueurs maximales en tokens : à reprendre dans DPOConfig (DPO_MAX_LENGTH de sft_lora.ipynb)
MAX_PROMPT_TOKENS = 512
MAX_LENGTH = 1_024  # prompt + la plus longue des deux réponses

MAX_PAIRS = 800      # plafond train + validation (moitié chosen plus longue, moitié plus courte)
MAX_ANSWER_CHARS = 2_000  # train + validation : réponses plus longues écartées (style trop long pour le français)

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chargement

Les trois parties : `train` et `validation` fournissent les paires d'entraînement, `test` (revu par des experts) le
jeu de test.

In [2]:
ds = load_dataset("TsinghuaC3I/UltraMedical-Preference")


def flatten(d: pd.DataFrame) -> pd.DataFrame:
    # Une ligne par paire : question, deux réponses, leurs modèles et leurs notes GPT-4, corrigé
    meta = pd.json_normalize(d["metadata"].tolist())
    return pd.DataFrame({
        "prompt_id": d["prompt_id"],
        "source": d["prompt_id"].str.split(",").str[0],  # "MedQA,1234" -> "MedQA"
        "label_type": d["label_type"],
        "prompt": d["prompt"].str.strip(),
        "chosen": d["chosen"].map(lambda messages: messages[-1]["content"].strip()),
        "rejected": d["rejected"].map(lambda messages: messages[-1]["content"].strip()),
        "chosen_model": meta["chosen.model"],
        "rejected_model": meta["rejected.model"],
        "chosen_score": meta["chosen.score"],
        "rejected_score": meta["rejected.score"],
        "golden_answer": meta["golden_answer"],  # lettre pour les QCM, texte libre ou vide sinon
    })


pool = pd.concat([flatten(ds[name].to_pandas()) for name in ("train", "validation")], ignore_index=True)
test = flatten(ds["test"].to_pandas())
print(f"train + validation : {len(pool)} paires ; test : {len(test)} paires")

train + validation : 111585 paires ; test : 777 paires


# Analyse

In [3]:
# Types de paires (« human » : préférences revues par des experts, uniquement dans le test)
pd.concat({"train + validation": pool["label_type"].value_counts(), "test": test["label_type"].value_counts()}, axis=1).fillna(0).astype(int)

,train + validation,test
label_type,,
hard,46690,196
length,41989,180
easy,22906,238
human,0,163


In [4]:
# Écart de note et longueur selon le type de paire : « easy » = grand écart, mais chosen presque toujours la plus longue
pool.assign(score_gap=pool["chosen_score"] - pool["rejected_score"],
            chosen_longer=pool["chosen"].str.len() > pool["rejected"].str.len()).groupby("label_type").agg(
    paires=("prompt", "size"), score_gap=("score_gap", "mean"), chosen_longer=("chosen_longer", "mean")).round(2)

,paires,score_gap,chosen_longer
label_type,,,
easy,22906,1.28,0.90
hard,46690,0.98,0.78
length,41989,1.36,0.38


# Exclusions

- **Questions du test** présentes aussi en train / validation (même texte) : retirées du pool.
- **Questions du SFT** : seules les questions attribuées au DPO par leur empreinte sont gardées.
- **Sources** : celles du SFT anglais (`SOURCES`).

In [5]:
pool = pool[~pool["prompt"].isin(set(test["prompt"]))]
pool["case_id"] = pool["prompt"].map(case_id)
pool = pool[pool["case_id"].map(lambda cid: case_assignment(cid)[0]) == "dpo"]
pool = pool[pool["source"].isin(SOURCES)]
print(f"pool : {len(pool)} paires, {pool['prompt'].nunique()} questions")
pool["source"].value_counts()

pool : 24772 paires, 17620 questions


source
MedMCQA       8812
ChatDoctor    5714
MedQA         5611
MedQA-Evol    4635
Name: count, dtype: int64

# Vérification des préférences

- **QCM** : la `chosen` doit donner la lettre du corrigé, et la `rejected` une autre lettre. Les paires où les deux
  réponses sont justes ne portent que sur la rédaction : écartées.
- **Sans corrigé** (ChatDoctor) : `chosen` très bien notée (≥ `MIN_CHOSEN_SCORE`) et écart net avec la `rejected`
  (≥ `MIN_SCORE_GAP`).
- **Nettoyage** : réponses vides ou identiques écartées.

La colonne `pair_check` garde la trace du contrôle appliqué.

In [6]:
ANSWER = re.compile(r"answer is:?\s*\(?\**([A-J])\b", re.I)  # « So, the answer is C. »


def answer_letter(text: str) -> str | None:
    found = ANSWER.findall(text)
    return found[-1].upper() if found else None


chosen_letter, rejected_letter = pool["chosen"].map(answer_letter), pool["rejected"].map(answer_letter)
is_mcq = pool["source"].isin(MCQ_SOURCES)
verified = is_mcq & (chosen_letter == pool["golden_answer"]) & rejected_letter.notna() & (rejected_letter != pool["golden_answer"])
clear = ~is_mcq & (pool["chosen_score"] >= MIN_CHOSEN_SCORE) & (pool["chosen_score"] - pool["rejected_score"] >= MIN_SCORE_GAP)
valid = (pool["chosen"] != "") & (pool["rejected"] != "") & (pool["chosen"] != pool["rejected"])
pool = pool[(verified | clear) & valid].assign(pair_check=lambda d: d["source"].isin(MCQ_SOURCES).map({True: "golden_answer", False: "gpt4_score"}))
print(f"paires vérifiées : {len(pool)}, {pool['prompt'].nunique()} questions")
pd.crosstab(pool["source"], pool["label_type"], margins=True)

paires vérifiées : 4195, 3406 questions


label_type,easy,hard,length,All
source,,,,
ChatDoctor,152,0,357,509
MedMCQA,380,288,1122,1790
MedQA,251,144,798,1193
MedQA-Evol,149,87,467,703
All,932,519,2744,4195


# Orientation triage

Mêmes critères que le SFT (`projet_14.selection`) : **patient décrit** obligatoire (vignette ou patient qui parle de
lui), puis un score de 0 à 2 (**constantes** dans la question, vocabulaire d'**urgence** dans la question ou la
`chosen`), qui donne la priorité lors de l'échantillonnage.

In [7]:
def add_triage(d: pd.DataFrame) -> pd.DataFrame:
    flags = [triage_flags(p, p + "\n" + c, "en") for p, c in zip(d["prompt"], d["chosen"])]
    return d.assign(patient=d["prompt"].str.contains(PATIENT_EN), constantes=[f[0] for f in flags],
                    urgence=[f[1] for f in flags]).assign(score=lambda x: x["constantes"].astype(int) + x["urgence"].astype(int))


pool = add_triage(pool)
pool = pool[pool["patient"]]
print(f"patient décrit : {len(pool)} paires, {pool['prompt'].nunique()} questions")
pd.crosstab(pool["source"], pool["score"], margins=True)

patient décrit : 2431 paires, 1974 questions


/tmp/ipykernel_13084/3656950853.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return d.assign(patient=d["prompt"].str.contains(PATIENT_EN), constantes=[f[0] for f in flags],


score,0,1,2,All
source,,,,
ChatDoctor,235,158,14,407
MedMCQA,177,97,19,293
MedQA,355,392,356,1103
MedQA-Evol,422,170,36,628
All,1189,817,425,2431


# Filtre sur la longueur (tokens)

Le DPO garde les deux réponses de chaque paire en mémoire : prompt ≤ `MAX_PROMPT_TOKENS` et prompt + la plus longue
réponse ≤ `MAX_LENGTH` (RTX 3080, 10 Go). Une paire coupée comparerait des réponses tronquées.

Pour l'entraînement (pas pour le test), les deux réponses font au plus `MAX_ANSWER_CHARS` caractères : le DPO
additionne les tokens, et les réponses très longues (surtout ChatDoctor) dominaient l'apprentissage.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def n_tokens(texts: pd.Series) -> list[int]:
    return [len(ids) for ids in tokenizer(texts.tolist(), add_special_tokens=False)["input_ids"]]


def fits(d: pd.DataFrame) -> pd.DataFrame:
    d = d.assign(n_prompt=n_tokens(d["prompt"]), n_chosen=n_tokens(d["chosen"]), n_rejected=n_tokens(d["rejected"]))
    keep = (d["n_prompt"] <= MAX_PROMPT_TOKENS) & (d["n_prompt"] + d[["n_chosen", "n_rejected"]].max(axis=1) <= MAX_LENGTH)
    print(f"{len(d)} paires, {(~keep).sum()} trop longues")
    return d[keep]


pool = fits(pool)
too_long = pool[["chosen", "rejected"]].apply(lambda answers: answers.str.len()).max(axis=1) > MAX_ANSWER_CHARS
print(f"réponse de plus de {MAX_ANSWER_CHARS} caractères : {too_long.sum()} paires écartées "
      f"({pool.loc[too_long, 'source'].value_counts().to_dict()})")
pool = pool[~too_long]
pool["chosen_longer"] = pool["n_chosen"] > pool["n_rejected"]
print(f"questions avec une paire où la chosen est la plus longue : {pool.loc[pool['chosen_longer'], 'prompt'].nunique()} ; "
      f"la plus courte : {pool.loc[~pool['chosen_longer'], 'prompt'].nunique()}")

2431 paires, 17 trop longues
réponse de plus de 2000 caractères : 675 paires écartées ({'ChatDoctor': 381, 'MedQA': 148, 'MedQA-Evol': 92, 'MedMCQA': 54})
questions avec une paire où la chosen est la plus longue : 461 ; la plus courte : 1020


# Échantillonnage équilibré et répartition

- **Une paire par question**, la plus nette (plus grand écart de note).
- **Équilibre sur la longueur** : autant de questions où la `chosen` est la plus longue que la plus courte. Les paires
  `easy` (chosen presque toujours la plus longue, préférence nette) compensent les paires `length` (chosen surtout la
  plus courte) : la longueur ne permet plus de deviner la préférence.
- **Priorité au triage** : dans chaque moitié, les questions de meilleur score passent d'abord, puis un ordre
  pseudo-aléatoire fixe tiré de l'identifiant.
- **Train / validation** : tirés de l'empreinte de la question (90 / 10). Le test est le jeu revu par des experts.

In [9]:
pool = pool.assign(score_gap=pool["chosen_score"] - pool["rejected_score"],
                   order=pool["case_id"].map(lambda cid: int(cid[:6], 16)))  # 1er tirage (le split vient du 2e : indépendant)
pool = pool.sort_values(["score", "order", "score_gap"], ascending=[False, True, False])


def best_pairs(d: pd.DataFrame) -> pd.DataFrame:
    # Une paire par question (la plus nette), dans l'ordre de priorité
    return d.loc[d.groupby("prompt", sort=False)["score_gap"].idxmax()].sort_values(["score", "order"], ascending=[False, True])


longer = best_pairs(pool[pool["chosen_longer"]])
shorter = best_pairs(pool[~pool["chosen_longer"] & ~pool["prompt"].isin(longer["prompt"])])
k = min(MAX_PAIRS // 2, len(longer), len(shorter))
longer = longer.iloc[:k]
shorter = best_pairs(pool[~pool["chosen_longer"] & ~pool["prompt"].isin(longer["prompt"])]).iloc[:k]
pairs = pd.concat([longer, shorter])

# Répartition par l'empreinte de la question : part « validation » du tirage -> validation, le reste -> train
pairs["split"] = pairs["case_id"].map(lambda cid: "validation" if case_assignment(cid)[1] == "validation" else "train")
print(f"{len(pairs)} paires ({k} chosen plus longue, {k} plus courte)")
pd.crosstab([pairs["split"], pairs["chosen_longer"]], pairs["pair_check"], margins=True)

800 paires (400 chosen plus longue, 400 plus courte)


pair_check                golden_answer  gpt4_score  All
split      chosen_longer                                
train      False                    357           0  357
           True                     361           9  370
validation False                     43           0   43
           True                      29           1   30
All                                 790          10  800

In [10]:
# Répartition par source, type de paire et score de triage
pd.concat({"source": pairs["source"].value_counts(), "label_type": pairs["label_type"].value_counts(),
           "score": pairs["score"].value_counts()}).to_frame("paires")

paires
source     MedQA          516
           MedQA-Evol     192
           MedMCQA         82
           ChatDoctor      10
label_type length         531
           easy           185
           hard            84
score      1              341
           2              285
           0              174

# Test revu par des experts

Gardé **entier** (toutes sources, tous types de paires) : c'est la distribution validée par des experts, non
équilibrée. Seuls le nettoyage et le filtre sur la longueur s'appliquent. La colonne `triage` (patient décrit et
score ≥ 1) permet de mesurer aussi la précision sur le sous-ensemble proche du triage.

In [11]:
test = test[(test["chosen"] != "") & (test["rejected"] != "") & (test["chosen"] != test["rejected"])].drop_duplicates(subset="prompt")
test = fits(add_triage(test)).assign(case_id=lambda d: d["prompt"].map(case_id), pair_check="expert_review", split="test")
test["chosen_longer"] = test["n_chosen"] > test["n_rejected"]
pairs["triage"] = pairs["score"] >= 1
test["triage"] = test["patient"] & (test["score"] >= 1)
print(f"test : {len(test)} paires, chosen plus longue dans {test['chosen_longer'].mean():.0%}, triage : {test['triage'].sum()}")

/tmp/ipykernel_13084/3656950853.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return d.assign(patient=d["prompt"].str.contains(PATIENT_EN), constantes=[f[0] for f in flags],


776 paires, 20 trop longues
test : 756 paires, chosen plus longue dans 70%, triage : 160


# Paires au format conversationnel

In [12]:
METADATA = ["prompt_id", "source", "case_id", "label_type", "pair_check", "golden_answer", "chosen_model", "rejected_model",
            "chosen_score", "rejected_score", "constantes", "urgence", "triage", "lang"]


def to_dpo(d: pd.DataFrame) -> pd.DataFrame:
    return d.assign(
        lang="en",
        golden_answer=d["golden_answer"].map(lambda g: g if isinstance(g, str) else ""),
        prompt=d["prompt"].map(lambda p: [{"role": "system", "content": SYSTEM_PROMPT_EN}, {"role": "user", "content": p}]),
        chosen=d["chosen"].map(lambda a: [{"role": "assistant", "content": a}]),
        rejected=d["rejected"].map(lambda a: [{"role": "assistant", "content": a}]),
    )


final_splits = {"train": pairs[pairs["split"] == "train"], "validation": pairs[pairs["split"] == "validation"], "test": test}
final_splits = {name: to_dpo(d) for name, d in final_splits.items()}
example = final_splits["train"].iloc[0]
print(f"--- PROMPT\n{example['prompt'][1]['content'][:600]}\n--- CHOSEN\n{example['chosen'][0]['content'][:400]}\n--- REJECTED\n{example['rejected'][0]['content'][:400]}")

--- PROMPT
A 60-year-old woman and her son are visited at her home by a health aid. He is her caregiver but has difficulty getting her out of the house. Her son is concerned about continuous and repetitive mouth and tongue movements that started about 2 weeks ago and have become more evident ever since. She is non-verbal at baseline and can complete most activities of daily living. She suffers from an unspecified psychiatric disorder. Her medications include fluphenazine. Today, her heart rate is 90/min, respiratory rate is 17/min, blood pressure is 125/87 mm Hg, and temperature is 37.0°C (98.6°F). On ph
--- CHOSEN
The patient is exhibiting continuous and repetitive mouth and tongue movements, which are likely symptoms of tardive dyskinesia (TD), a known side effect of long-term use of antipsychotic medications such as fluphenazine. TD is a movement disorder that can occur in patients taking antipsychotics, especially in older adults.

Given that the patient's symptoms started about 2 

# Export JSONL

In [13]:
OUT.mkdir(parents=True, exist_ok=True)
for name, d in final_splits.items():
    Dataset.from_pandas(d[METADATA + ["prompt", "chosen", "rejected"]], preserve_index=False).to_json(
        OUT / f"{STEM}_{name}.jsonl", force_ascii=False)
    print(name, len(d), "paires")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 33.26ba/s]

train 727 paires


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 166.16ba/s]

validation 73 paires


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 22.80ba/s]

test 756 paires


# Anonymisation des données

Seule la source **ChatDoctor** contient de vrais échanges entre patients et médecins, avec des noms (« Hello
Shonda »). Elle est anonymisée en entier (prompt, chosen, rejected). Les autres sources (questions d'examen,
manuels, Wikipédia, réponses générées) n'ont pas de donnée personnelle réelle ; sur elles, le modèle spaCy anglais ne
produit presque que des faux positifs (gènes, éponymes, options de QCM).

In [14]:
reports = {
    name: run(OUT / f"{STEM}_{name}.jsonl", lang="en", output_dir=OUT / "anonymized", sources=["ChatDoctor"])
    for name in ("train", "validation", "test")
}
pd.DataFrame(reports).T[["records", "records_scanned", "records_modified", "persons_replaced"]]

,records,records_scanned,records_modified,persons_replaced
train,727,9,3,3
validation,73,1,1,2
test,756,92,53,131


# Contrôle final

In [15]:
final = load_dataset("json", data_files={s: str(OUT / "anonymized" / f"{STEM}_{s}.anonymized.jsonl") for s in ("train", "validation", "test")})
print(final)

# Aucune paire avec chosen == rejected
for name, split in final.items():
    assert not any(c[0]["content"] == r[0]["content"] for c, r in zip(split["chosen"], split["rejected"])), name

# Aucune question partagée entre les splits, ni avec le SFT anglais (prep_SFT_UltraMedical, à exécuter avant)
ids = {s: set(d["case_id"]) for s, d in final.items()}
assert not ids["train"] & (ids["validation"] | ids["test"]) and not ids["validation"] & ids["test"], "question partagée entre splits"
sft = load_dataset("json", data_files={s: str(SFT_DIR / f"sft_ultramedical_{s}.anonymized.jsonl") for s in ("train", "validation", "test")})
sft_ids = set().union(*(set(d["case_id"]) for d in sft.values()))
assert not sft_ids & set().union(*ids.values()), "question commune avec le SFT"
print("OK : aucune paire identique, aucune question partagée entre les splits ni avec le SFT")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 727 examples [00:00, 49373.52 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 73 examples [00:00, 12590.85 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 756 examples [00:00, 20748.39 examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'source', 'case_id', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'constantes', 'urgence', 'triage', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 727
    })
    validation: Dataset({
        features: ['prompt_id', 'source', 'case_id', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'constantes', 'urgence', 'triage', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 73
    })
    test: Dataset({
        features: ['prompt_id', 'source', 'case_id', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'constantes', 'urgence', 'triage', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 756
    })
})


OK : aucune paire identique, aucune question partagée entre les splits ni avec le SFT


# Limites

- **Préférences** : vérifiées par le corrigé pour les QCM (`golden_answer`), jugées par GPT-4 seul pour ChatDoctor
  (`gpt4_score`, biais possible en faveur des réponses de GPT-4). Seul le test a été revu par des experts.
- **Test non équilibré** sur la longueur (`chosen` plus longue dans la majorité des paires) : l'évaluation DPO le
  sépare selon la longueur pour détecter un biais restant.
- **Anglais uniquement** : la partie française du DPO reste à constituer (MediQAl, paires construites à partir du
  corrigé, sur des cas non retenus pour le SFT).
- **Contexte américain** (USMLE) : unités, médicaments et organisation des soins diffèrent parfois des protocoles français.